In [1]:
#문항1
import re
import datetime

In [2]:
PATTERNS = [
    
    ("ymd_kor",   re.compile(r"(?P<y>\d{4})\s*년\s*(?P<m>\d{1,2})\s*월\s*(?P<d>\d{1,2})\s*일")),

    ("ymd_sep",   re.compile(r"(?P<y>\d{4})[.\-/](?P<m>\d{1,2})[.\-/](?P<d>\d{1,2})")),

    ("mdy_slash", re.compile(r"(?P<m>\d{1,2})/(?P<d>\d{1,2})/(?P<y>\d{4})")),

    ("ymd_short", re.compile(r"(?<!\d)(?P<y>\d{2})[.\-/](?P<m>\d{1,2})[.\-/](?P<d>\d{1,2})(?!\d)")),
]


In [ ]:
def normalize_date(s):
    if not s or not str(s).strip():
        return (None, "빈 값")

    for tag, pat in PATTERNS:
        m = pat.search(str(s))
        if not m:
            continue

        y, mo, d = int(m["y"]), int(m["m"]), int(m["d"])
        if tag == "ymd_short":
            y += 2000 if y < 70 else 1900        

        try:
            dt = datetime.date(y, mo, d).strftime('%Y-%m-%d')
        except ValueError:
            return (None, f"{tag} · 유효하지 않은 날짜")

        return (dt, tag)

    return (None, "매치 없음")

In [5]:
samples = [
    "2024.12.24", "2024-12-24", "2024/12/24",
    "24.12.24", "2024년 12월 24일", "2024년 3월 5일",
    "12/24/2024", "2024.12.24 14:30", "등록일 : 2024.12.24",
    "2024-13-45", "작성일 없음", "",
]
for s in samples:
    value, tag = normalize_date(s)
    print(f"{s:22s} → {str(value):12s} [{tag}]")

2024.12.24             → 2024-12-24   [ymd_sep]
2024-12-24             → 2024-12-24   [ymd_sep]
2024/12/24             → 2024-12-24   [ymd_sep]
24.12.24               → 2024-12-24   [ymd_short]
2024년 12월 24일          → 2024-12-24   [ymd_kor]
2024년 3월 5일            → 2024-03-05   [ymd_kor]
12/24/2024             → 2024-12-24   [mdy_slash]
2024.12.24 14:30       → 2024-12-24   [ymd_sep]
등록일 : 2024.12.24       → 2024-12-24   [ymd_sep]
2024-13-45             → None         [ymd_sep · 유효하지 않은 날짜]
작성일 없음                 → None         [매치 없음]
                       → None         [빈 값]


In [49]:
#문항2
from collections import Counter
import re
import pandas as pd

In [ ]:
LOG = re.compile(
    r"^(?P<ip>\d{1,3}(?:\.\d{1,3}){3})\s+"
    r"\S+\s+\S+\s+"
    r"\[(?P<timestamp>[^\]]+)\]\s+"
    r'"(?P<method>[A-Z]+)\s+(?P<path>\S+)[^"]*"\s+'
    r"(?P<status>\d{3})\s+"
    r"(?P<bytes>\d+)\s+"
    r'"(?P<user_agent>[^"]*)"'
)

In [47]:
BOT = re.compile("bot|crawler|spider|python-requests", re.IGNORECASE)

def parse_log(path: str) -> tuple[pd.DataFrame, int]:
  rows, skipped = [], 0
  with open(path, encoding="utf-8") as f:
    for line in f.readlines():
      line = line.strip()
      if not line:
        continue
      m = LOG.match(line)
      if not m:
        skipped += 1
        continue
      rows.append(m.groupdict())

  df = pd.DataFrame(rows)
  if len(df) > 0:
    df["status"] = df["status"].astype(int)
    df["bytes"] = df["bytes"].astype(int)
    df["is_bot"] = df["user_agent"].str.contains(BOT, na=False)
  return df, skipped


In [48]:
df, skipped = parse_log("access.log")
print(f"파싱 {len(df)}줄 · 건너뜀 {skipped}줄")

print("\n── 상태코드별 요청 수 ──")
print(df["status"].value_counts().sort_index().to_string())

print("\n── 4xx·5xx 발생 경로 상위 5 ──")
bad = df[df["status"] >= 400]
print(bad["path"].value_counts().head(5).to_string() if not bad.empty else " 없음")

print("\n── 봇 의심 User-Agent ──")
bots = df[df["is_bot"]]["user_agent"].value_counts()
print(bots.to_string() if not bots.empty else " 없음")
print(f" 봇 요청 비율 {df['is_bot'].mean() * 100:.1f}%")

df.to_csv("access_report.csv", index=False, encoding="utf-8-sig")

파싱 3줄 · 건너뜀 0줄

── 상태코드별 요청 수 ──
status
200    1
404    1
500    1

── 4xx·5xx 발생 경로 상위 5 ──
path
/error.html    1
/login         1

── 봇 의심 User-Agent ──
user_agent
python-requests/2.31.0    1
Googlebot/2.1             1
 봇 요청 비율 66.7%
